# Mask R-CNN Training on Pre-SAM'd Dataset (Anti-Overfit)

**Optimised for small datasets** — 51 train + 11 val original images, augmented to ~10K crops.

**Anti-overfit strategy:**
- **Train/Val split** based on original source images (val crops never seen during training)
- **2-phase training**: Phase 1 = freeze backbone (5 epochs), Phase 2 = unfreeze all (20 epochs)
- **Strong augmentation** (elastic, affine, color jitter, cutout) — only on train
- **Early stopping** (patience 7) on validation loss
- **ReduceLROnPlateau** scheduler — data-driven LR decay
- **Higher weight decay** (0.005) + gradient clipping
- Only model files saved to Drive

**Source:** `MyDrive/mp-detect/data/yolo-augmented-sam/` (images + masks + annotations.json)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
drive_data = '/content/drive/MyDrive/mp-detect/data/yolo-augmented-sam'
assert os.path.isdir(drive_data), f"Dataset not found: {drive_data}"
print(f"\u2713 Drive mounted, dataset found at {drive_data}")

## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision albumentations pycocotools tqdm matplotlib

In [ ]:
import json
import random
import time
import shutil
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision import transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("\u2713 All imports OK")

## 3. Embedded Functions

In [ ]:
# ============================================================================
# EMBEDDED FUNCTIONS — anti-overfit edition
# ============================================================================

NUM_CLASSES = 4
CLASS_NAMES = ['background', 'fiber', 'film', 'fragment']
YOLO_TO_MASKRCNN = {0: 1, 1: 2, 2: 3}


class CropDataset(Dataset):
    """Dataset of SAM-annotated crops for Mask R-CNN training.

    Args:
        crops_dir:    Path to directory with images/, masks/, annotations.json
        transforms:   Albumentations pipeline
        split_filter: If given ('train' or 'val'), only keep samples whose
                      annotation has 'split' == split_filter.
        sample_keys:  If given, use only these annotation keys (overrides split_filter).
    """

    CLASS_NAME_TO_YOLO_ID = {'fiber': 0, 'film': 1, 'fragment': 2}

    def __init__(self, crops_dir: str, transforms=None,
                 split_filter: str = None, sample_keys: list = None):
        self.crops_dir = Path(crops_dir)
        self.transforms = transforms

        ann_file = self.crops_dir / 'annotations.json'
        has_flat_images = (self.crops_dir / 'images').is_dir()
        has_class_dirs = any((self.crops_dir / c).is_dir()
                            for c in self.CLASS_NAME_TO_YOLO_ID)

        if ann_file.exists():
            with open(ann_file) as f:
                self.annotations = json.load(f)
            if has_flat_images:
                self.images_dir = self.crops_dir / 'images'
            elif has_class_dirs:
                self.images_dir = None
            else:
                self.images_dir = self.crops_dir / 'images'
        elif has_class_dirs:
            self.annotations = {}
            self.images_dir = None
            for cls_name, cls_id in self.CLASS_NAME_TO_YOLO_ID.items():
                cls_dir = self.crops_dir / cls_name
                if not cls_dir.is_dir():
                    continue
                for img_file in sorted(cls_dir.glob('*.png')):
                    crop = cv2.imread(str(img_file))
                    if crop is None:
                        continue
                    h, w = crop.shape[:2]
                    self.annotations[img_file.name] = {
                        'source_image': '', 'class_id': cls_id,
                        'class_name': cls_name, 'yolo_confidence': 1.0,
                        'rel_box': [0, 0, w, h], 'crop_size': [w, h]
                    }
            print(f"Auto-generated annotations for {len(self.annotations)} crops")
        else:
            raise FileNotFoundError(
                f"No annotations.json or class subdirs in {crops_dir}")

        # --- Filter by split or explicit keys ---
        if sample_keys is not None:
            self.samples = [k for k in sample_keys if k in self.annotations]
        elif split_filter:
            self.samples = [
                k for k, v in self.annotations.items()
                if v.get('split', 'train') == split_filter
            ]
        else:
            self.samples = list(self.annotations.keys())

        print(f"Loaded {len(self.samples)} samples"
              + (f" (split={split_filter})" if split_filter else ""))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample_name = self.samples[idx]
        ann = self.annotations[sample_name]

        if self.images_dir is not None:
            img_path = self.images_dir / sample_name
        else:
            cls_name = ann.get('class_name', '')
            img_path = self.crops_dir / cls_name / sample_name

        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f"Could not load: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        # Load mask — prefer SAM mask, fall back to ellipse
        mask = None
        mask_file = ann.get('mask_file')
        masks_dir = self.crops_dir / 'masks'

        if mask_file and (masks_dir / mask_file).exists():
            raw = cv2.imread(str(masks_dir / mask_file), cv2.IMREAD_GRAYSCALE)
            if raw is not None:
                mask = (raw > 127).astype(np.uint8)
        if mask is None:
            default_mask = masks_dir / sample_name.replace('.png', '_mask.png')
            if default_mask.exists():
                raw = cv2.imread(str(default_mask), cv2.IMREAD_GRAYSCALE)
                if raw is not None:
                    mask = (raw > 127).astype(np.uint8)
        if mask is None:
            mask = self._create_ellipse_mask(h, w, ann.get('rel_box'))
        if mask.shape[:2] != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

        ys, xs = np.where(mask > 0)
        if len(xs) > 0 and len(ys) > 0:
            box = [xs.min(), ys.min(), xs.max(), ys.max()]
        else:
            margin = min(h, w) // 10
            box = [margin, margin, w - margin, h - margin]

        class_id = YOLO_TO_MASKRCNN[ann['class_id']]
        boxes = np.array([box], dtype=np.float32)
        labels = np.array([class_id], dtype=np.int64)
        masks = np.array([mask], dtype=np.uint8)

        if self.transforms:
            transformed = self.transforms(
                image=image, bboxes=boxes.tolist(),
                masks=list(masks), class_labels=labels.tolist()
            )
            image = transformed['image']
            if len(transformed['bboxes']) > 0:
                boxes = np.array(transformed['bboxes'], dtype=np.float32)
                labels = np.array(transformed['class_labels'], dtype=np.int64)
                masks = np.array(transformed['masks'], dtype=np.uint8)
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0

        target = {
            'boxes': torch.as_tensor(boxes, dtype=torch.float32),
            'labels': torch.as_tensor(labels, dtype=torch.int64),
            'masks': torch.as_tensor(masks, dtype=torch.uint8),
            'image_id': torch.tensor([idx]),
            'area': torch.as_tensor(
                [(b[2]-b[0])*(b[3]-b[1]) for b in boxes], dtype=torch.float32),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return image, target

    def _create_ellipse_mask(self, h, w, rel_box=None):
        mask = np.zeros((h, w), dtype=np.uint8)
        if rel_box:
            x1, y1, x2, y2 = rel_box
            center = ((x1+x2)//2, (y1+y2)//2)
            axes = ((x2-x1)//2, (y2-y1)//2)
        else:
            center = (w//2, h//2)
            axes = (int(w*0.4), int(h*0.4))
        if axes[0] > 0 and axes[1] > 0:
            cv2.ellipse(mask, center, axes, 0, 0, 360, 1, -1)
        return mask


def get_transforms(train=True, img_size=128):
    """Augmentation pipelines.  Train is deliberately aggressive to
    fight overfitting on a heavily-augmented small dataset."""
    if train:
        return A.Compose([
            A.Resize(img_size, img_size),
            # --- spatial ---
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15,
                               rotate_limit=30, p=0.5),
            A.ElasticTransform(alpha=30, sigma=5, p=0.2),
            # --- colour / noise ---
            A.RandomBrightnessContrast(brightness_limit=0.3,
                                       contrast_limit=0.3, p=0.5),
            A.HueSaturationValue(hue_shift_limit=10,
                                 sat_shift_limit=20,
                                 val_shift_limit=20, p=0.3),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),
            # --- cutout (dropout rectangles) ---
            A.CoarseDropout(max_holes=4, max_height=img_size//8,
                            max_width=img_size//8, p=0.3),
            # --- normalize ---
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ], bbox_params=A.BboxParams(
            format='pascal_voc', label_fields=['class_labels'],
            min_visibility=0.3))
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ], bbox_params=A.BboxParams(
            format='pascal_voc', label_fields=['class_labels'],
            min_visibility=0.3))


def collate_fn(batch):
    return tuple(zip(*batch))


def get_model(num_classes: int, pretrained: bool = True):
    if pretrained:
        model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    else:
        model = maskrcnn_resnet50_fpn(weights=None)
    in_f = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes)
    in_fm = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_fm, 256, num_classes)
    return model


def freeze_backbone(model, freeze=True):
    """Freeze / unfreeze the ResNet-50 + FPN backbone."""
    for p in model.backbone.parameters():
        p.requires_grad = not freeze
    status = "frozen" if freeze else "unfrozen"
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Backbone {status}  ({n:,} trainable params)")


@torch.no_grad()
def evaluate(model, loader, device):
    """Run model in train mode on val data to get loss (no grad)."""
    model.train()                         # need train mode for loss
    total_loss = 0.0
    component_losses = defaultdict(float)
    count = 0
    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        if not all(len(t['boxes']) > 0 for t in targets):
            continue
        loss_dict = model(images, targets)
        total_loss += sum(v.item() for v in loss_dict.values())
        for k, v in loss_dict.items():
            component_losses[k] += v.item()
        count += 1
    avg = total_loss / max(count, 1)
    comp = {k: v / max(count, 1) for k, v in component_losses.items()}
    return avg, comp


print("\u2713 All functions defined (anti-overfit edition)")

## 4. Copy Dataset to Local Colab Storage

Copy the SAM dataset from Drive to Colab's local SSD. This is resilient to Drive disconnects — it copies file-by-file with retries and auto-remounts. Re-run this cell to resume if interrupted.

In [ ]:
# ============================================================================
# Resilient Copy: Drive → Local SSD (file-by-file, retry + auto-remount)
# ============================================================================

# --- Source (Google Drive) ---
DRIVE_ROOT = Path('/content/drive/MyDrive/mp-detect')
DRIVE_SAM_DIR = DRIVE_ROOT / 'data' / 'yolo-augmented-sam'

# --- Destination (local Colab SSD, fast, no disconnects) ---
LOCAL_ROOT = Path('/content/mp_data')
SAM_DIR = LOCAL_ROOT / 'yolo-augmented-sam'
SAVE_DIR = LOCAL_ROOT / 'experiments'

# --- Drive destination (only for final model files) ---
DRIVE_EXPERIMENTS = DRIVE_ROOT / 'experiments'


def _remount_drive():
    """Force-remount Google Drive after a disconnect."""
    print("  \u21bb Remounting Google Drive...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        time.sleep(2)
    except Exception:
        pass
    from google.colab import drive as _d
    _d.mount('/content/drive', force_remount=True)
    time.sleep(3)
    print("  \u2713 Drive remounted")


def robust_copy_file(src: Path, dst: Path, max_retries=3):
    dst.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, max_retries + 1):
        try:
            shutil.copy2(str(src), str(dst))
            return True
        except OSError as e:
            if e.errno == 107 or 'Transport endpoint' in str(e):
                print(f"  \u26a0 Drive disconnected copying {src.name} "
                      f"(attempt {attempt}/{max_retries})")
                _remount_drive()
            else:
                raise
    print(f"  \u2717 FAILED after {max_retries} retries: {src.name}")
    return False


def robust_copytree(src_dir: Path, dst_dir: Path):
    src_dir, dst_dir = Path(src_dir), Path(dst_dir)
    copied, skipped, failed = 0, 0, 0
    files_only = [f for f in src_dir.rglob('*') if f.is_file()]
    print(f"  Found {len(files_only)} files to copy")
    for src_file in files_only:
        rel = src_file.relative_to(src_dir)
        dst_file = dst_dir / rel
        if dst_file.exists() and dst_file.stat().st_size > 0:
            skipped += 1
            continue
        ok = robust_copy_file(src_file, dst_file)
        if ok:
            copied += 1
        else:
            failed += 1
        total = copied + skipped + failed
        if total % 200 == 0:
            print(f"  ... {copied} copied, {skipped} skipped, "
                  f"{failed} failed / {len(files_only)}")
    print(f"  Done: {copied} copied, {skipped} existed, {failed} failed")
    return failed == 0


# Copy SAM dataset (resumable)
marker = SAM_DIR / '.copy_complete'
if not marker.exists():
    SAM_DIR.mkdir(parents=True, exist_ok=True)
    print("Copying SAM dataset to local storage (resilient, resumable)...")
    if robust_copytree(DRIVE_SAM_DIR, SAM_DIR):
        marker.touch()
        print(f"\u2713 Copy complete \u2192 {SAM_DIR}")
    else:
        print("\u26a0 Some files failed \u2014 re-run this cell to resume")
else:
    print(f"\u2713 SAM data already local: {SAM_DIR}")

SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================================
# Hyperparameters — tuned for 51 train + 11 val (heavily augmented to ~10K)
# ============================================================================
CROP_SIZE = 128
MASKRCNN_BATCH_SIZE = 8

# Phase 1: backbone frozen — warm up heads only
PHASE1_EPOCHS = 5
PHASE1_LR = 0.0005

# Phase 2: everything unfrozen — fine-tune
PHASE2_EPOCHS = 20
PHASE2_LR = 0.0001          # 5x smaller than phase 1

WEIGHT_DECAY = 0.005         # aggressive regularisation
EARLY_STOP_PATIENCE = 7     # stop if val loss stalls for 7 epochs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {DEVICE}")
print(f"SAM data:    {SAM_DIR}")
print(f"Save dir:    {SAVE_DIR}")
print(f"\nTraining plan:")
print(f"  Phase 1: {PHASE1_EPOCHS} epochs, LR={PHASE1_LR}, backbone FROZEN")
print(f"  Phase 2: {PHASE2_EPOCHS} epochs, LR={PHASE2_LR}, backbone UNFROZEN")
print(f"  Early stopping patience: {EARLY_STOP_PATIENCE}")
print(f"  Weight decay: {WEIGHT_DECAY}")

## 5. Explore Dataset

Inspect images, masks, class distribution, and sample overlays.

In [ ]:
sam_path = Path(SAM_DIR)

with open(sam_path / 'annotations.json') as f:
    annotations = json.load(f)

n_images = len(list((sam_path / 'images').glob('*.png')))
n_masks = len(list((sam_path / 'masks').glob('*.png')))
print(f"Images: {n_images}")
print(f"Masks:  {n_masks}")
print(f"Annotations: {len(annotations)}")

# Class distribution
class_counts = defaultdict(int)
for ann in annotations.values():
    class_counts[ann.get('class_name', 'unknown')] += 1

print(f"\nClass Distribution:")
for cls, count in sorted(class_counts.items()):
    pct = count / len(annotations) * 100
    print(f"  {cls:10s}: {count:5d} ({pct:.1f}%)")

# Show sample images with SAM masks
sample_keys = random.sample(list(annotations.keys()),
                            min(6, len(annotations)))

fig, axes = plt.subplots(2, 6, figsize=(24, 8))
fig.suptitle('SAM Dataset Samples  (top: original, bottom: mask overlay)',
             fontsize=14)

for j, name in enumerate(sample_keys):
    ann = annotations[name]
    img = cv2.imread(str(sam_path / 'images' / name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mask_file = ann.get('mask_file', name.replace('.png', '_mask.png'))
    mask_path = sam_path / 'masks' / mask_file
    mask = None
    if mask_path.exists():
        raw = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if raw is not None:
            mask = (raw > 127).astype(np.uint8)

    axes[0, j].imshow(img)
    axes[0, j].set_title(f"{ann.get('class_name','?')}", fontsize=9)
    axes[0, j].axis('off')

    if mask is not None:
        overlay = img.copy()
        overlay[mask == 1] = (overlay[mask == 1] * 0.5 +
                              np.array([0, 255, 0]) * 0.5).astype(np.uint8)
        axes[1, j].imshow(overlay)
    else:
        axes[1, j].imshow(img)
    score = ann.get('sam_score', None)
    title = f"mask" + (f" ({score:.3f})" if score else "")
    axes[1, j].set_title(title, fontsize=9)
    axes[1, j].axis('off')

plt.tight_layout()
plt.show()

## 6. Build Train / Val Datasets

Split by `source_image` origin — crops from val images are **never** seen during training.
If annotations don't have a `split` field, fall back to an 80/20 split by unique source images.

In [ ]:
# ---- Load all annotations & decide split strategy ----
sam_path = Path(SAM_DIR)
with open(sam_path / 'annotations.json') as f:
    all_annotations = json.load(f)

# Check if annotations have a 'split' field
has_split = any(v.get('split') for v in all_annotations.values())

if has_split:
    # Use existing split field (comes from YOLO train/val)
    train_keys = [k for k, v in all_annotations.items() if v.get('split') == 'train']
    val_keys   = [k for k, v in all_annotations.items() if v.get('split') == 'val']
    print(f"Using annotation 'split' field")
else:
    # Fall back: group by source_image, 80% train / 20% val
    source_to_keys = defaultdict(list)
    for k, v in all_annotations.items():
        src = v.get('source_image', k)
        source_to_keys[src].append(k)
    sources = sorted(source_to_keys.keys())
    random.seed(42)
    random.shuffle(sources)
    n_train = max(1, int(len(sources) * 0.8))
    train_sources = set(sources[:n_train])
    val_sources = set(sources[n_train:])
    train_keys = [k for s in train_sources for k in source_to_keys[s]]
    val_keys   = [k for s in val_sources   for k in source_to_keys[s]]
    print(f"No 'split' field — split by source image "
          f"({len(train_sources)} train / {len(val_sources)} val sources)")

print(f"\nTrain crops: {len(train_keys)}")
print(f"Val   crops: {len(val_keys)}")

# --- Class distribution per split ---
for label, keys in [('Train', train_keys), ('Val', val_keys)]:
    counts = defaultdict(int)
    for k in keys:
        counts[all_annotations[k].get('class_name', '?')] += 1
    print(f"\n  {label} class distribution:")
    for cls in sorted(counts):
        print(f"    {cls:10s}: {counts[cls]}")

# --- Build datasets ---
train_transforms = get_transforms(train=True, img_size=CROP_SIZE)
val_transforms   = get_transforms(train=False, img_size=CROP_SIZE)

train_dataset = CropDataset(str(SAM_DIR), transforms=train_transforms,
                            sample_keys=train_keys)
val_dataset   = CropDataset(str(SAM_DIR), transforms=val_transforms,
                            sample_keys=val_keys)

# Quick sanity check
sample_img, sample_target = train_dataset[0]
print(f"\nSample image shape: {sample_img.shape}")
print(f"Sample label: {CLASS_NAMES[sample_target['labels'][0].item()]}")

# Visualize augmented training samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Training Samples (Heavy Augmentation)', fontsize=14)
for i in range(8):
    idx = random.randint(0, len(train_dataset) - 1)
    img, target = train_dataset[idx]
    row, col = i // 4, i % 4
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    disp = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    mask = target['masks'][0].numpy()
    overlay = disp.copy()
    overlay[mask == 1] = overlay[mask == 1] * 0.5 + np.array([0, 1, 0]) * 0.5
    axes[row, col].imshow(overlay)
    axes[row, col].set_title(CLASS_NAMES[target['labels'][0].item()], fontsize=10)
    axes[row, col].axis('off')
plt.tight_layout()
plt.show()

## 7. Define Model & DataLoaders

Phase 1: backbone frozen → only train head layers (prevents catastrophic forgetting).
Phase 2: unfreeze all → fine-tune end-to-end with lower LR.

In [ ]:
# Create Mask R-CNN
model = get_model(NUM_CLASSES, pretrained=True)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Mask R-CNN (ResNet50-FPN)")
print(f"  Total parameters: {total_params:,}")
print(f"  Num classes:      {NUM_CLASSES} ({CLASS_NAMES})")
print(f"  Device:           {DEVICE}")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=MASKRCNN_BATCH_SIZE,
                          shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=MASKRCNN_BATCH_SIZE,
                          shuffle=False, num_workers=2, collate_fn=collate_fn)

print(f"\n  Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"  Val:   {len(val_dataset)} samples, {len(val_loader)} batches")

## 8. Train Mask R-CNN (2-Phase + Early Stopping)

- **Phase 1** (backbone frozen): head layers only, higher LR, quick convergence.
- **Phase 2** (all unfrozen): end-to-end fine-tune, lower LR, early stopping on val loss.

In [ ]:
# ============================================================================
# 2-PHASE TRAINING with validation & early stopping
# ============================================================================
best_val_loss = float('inf')
patience_counter = 0
global_epoch = 0

history = {
    'epoch': [], 'train_loss': [], 'val_loss': [], 'lr': [], 'phase': [],
    'loss_classifier': [], 'loss_box_reg': [],
    'loss_mask': [], 'loss_objectness': [], 'loss_rpn_box_reg': [],
    'val_loss_mask': [],
}

def run_phase(phase_name, n_epochs, lr, freeze_bb):
    global best_val_loss, patience_counter, global_epoch
    early_stopped = False

    # Freeze / unfreeze backbone
    freeze_backbone(model, freeze=freeze_bb)

    # Rebuild optimizer for this phase (new param groups)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True)

    print(f"\n{'='*60}")
    print(f"{phase_name}: {n_epochs} epochs, LR={lr}, "
          f"backbone={'FROZEN' if freeze_bb else 'UNFROZEN'}")
    print(f"{'='*60}\n")

    for ep in range(1, n_epochs + 1):
        global_epoch += 1
        model.train()
        epoch_loss = 0.0
        epoch_comp = defaultdict(float)
        batch_count = 0

        pbar = tqdm(train_loader,
                    desc=f"[{phase_name}] Epoch {ep}/{n_epochs}")
        for images, targets in pbar:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()}
                       for t in targets]
            if not all(len(t['boxes']) > 0 for t in targets):
                continue

            loss_dict = model(images, targets)
            losses = sum(v for v in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
            optimizer.step()

            bl = losses.item()
            epoch_loss += bl
            batch_count += 1
            for k, v in loss_dict.items():
                epoch_comp[k] += v.item()
            pbar.set_postfix({'loss': f'{bl:.4f}'})

        avg_train = epoch_loss / max(batch_count, 1)

        # --- Validation ---
        val_loss, val_comp = evaluate(model, val_loader, DEVICE)
        scheduler.step(val_loss)
        cur_lr = optimizer.param_groups[0]['lr']

        # --- Record history ---
        history['epoch'].append(global_epoch)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(val_loss)
        history['lr'].append(cur_lr)
        history['phase'].append(phase_name)
        for k in ['loss_classifier', 'loss_box_reg', 'loss_mask',
                   'loss_objectness', 'loss_rpn_box_reg']:
            history[k].append(epoch_comp.get(k, 0) / max(batch_count, 1))
        history['val_loss_mask'].append(val_comp.get('loss_mask', 0))

        print(f"  Epoch {ep}/{n_epochs}  "
              f"train={avg_train:.4f}  val={val_loss:.4f}  "
              f"val_mask={val_comp.get('loss_mask',0):.4f}  LR={cur_lr:.6f}")

        # --- Checkpoint ---
        ckpt = {
            'epoch': global_epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train,
            'val_loss': val_loss,
            'history': history,
        }
        torch.save(ckpt, str(SAVE_DIR / 'maskrcnn_crops_latest.pth'))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(ckpt, str(SAVE_DIR / 'maskrcnn_crops_best.pth'))
            print(f"    \u2192 New best model! (val_loss={val_loss:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"    \u2717 Early stopping triggered "
                      f"(patience {EARLY_STOP_PATIENCE})")
                early_stopped = True
                break

    return early_stopped

# ---- Phase 1: Backbone FROZEN ----
stopped = run_phase("Phase 1 (heads only)", PHASE1_EPOCHS,
                    PHASE1_LR, freeze_bb=True)

# ---- Phase 2: Full fine-tune (skip if already stopped) ----
if not stopped:
    run_phase("Phase 2 (full fine-tune)", PHASE2_EPOCHS,
              PHASE2_LR, freeze_bb=False)

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"  Total epochs:  {global_epoch}")
print(f"  Best val loss: {best_val_loss:.4f}")
print(f"{'='*60}")

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Mask R-CNN Training Curves (Train vs Val)', fontsize=14)

epochs = history['epoch']

# --- Total loss: train vs val ---
axes[0, 0].plot(epochs, history['train_loss'], 'b-', lw=2, label='Train')
axes[0, 0].plot(epochs, history['val_loss'], 'r-', lw=2, label='Val')
# Mark phase boundary
if PHASE1_EPOCHS < len(epochs):
    axes[0, 0].axvline(x=PHASE1_EPOCHS, color='gray', ls='--', alpha=0.5,
                        label='Unfreeze backbone')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Total Loss (train vs val)')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

# --- Component losses (train) ---
for key, color, label in [
    ('loss_classifier', 'r', 'Classifier'), ('loss_box_reg', 'g', 'Box Reg'),
    ('loss_mask', 'b', 'Mask'), ('loss_objectness', 'm', 'Objectness'),
    ('loss_rpn_box_reg', 'c', 'RPN Box Reg'),
]:
    if history.get(key):
        axes[0, 1].plot(epochs, history[key], color=color, label=label, lw=1.5)
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Train Component Losses')
axes[0, 1].legend(fontsize=8); axes[0, 1].grid(True, alpha=0.3)

# --- Learning rate ---
axes[1, 0].plot(epochs, history['lr'], 'g-', lw=2)
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('LR')
axes[1, 0].set_title('Learning Rate (ReduceLROnPlateau)')
axes[1, 0].grid(True, alpha=0.3)

# --- Mask loss: train vs val ---
if history.get('loss_mask') and history.get('val_loss_mask'):
    axes[1, 1].plot(epochs, history['loss_mask'], 'b-', lw=2, label='Train')
    axes[1, 1].plot(epochs, history['val_loss_mask'], 'r-', lw=2, label='Val')
    if PHASE1_EPOCHS < len(epochs):
        axes[1, 1].axvline(x=PHASE1_EPOCHS, color='gray', ls='--', alpha=0.5)
    axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Mask Loss')
    axes[1, 1].set_title('Mask Loss (Key Metric — train vs val)')
    axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Overfit diagnostic ---
final_train = history['train_loss'][-1]
final_val   = history['val_loss'][-1]
gap = final_val - final_train
print(f"\nFinal Metrics (Epoch {epochs[-1]}):")
print(f"  Train Loss: {final_train:.4f}")
print(f"  Val Loss:   {final_val:.4f}")
print(f"  Gap:        {gap:.4f} {'(OVERFIT \u26a0)' if gap > 0.5 else '(OK \u2713)'}")
print(f"  Best Val:   {best_val_loss:.4f}")

## 10. Visualize Predictions

In [ ]:
# Load best model
ckpt = torch.load(str(SAVE_DIR / 'maskrcnn_crops_best.pth'), map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']} "
      f"(loss: {ckpt['loss']:.4f})")

inference_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((CROP_SIZE, CROP_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

sam_path = Path(SAM_DIR)
with open(sam_path / 'annotations.json') as f:
    test_anns = json.load(f)

test_samples = random.sample(list(test_anns.keys()), min(8, len(test_anns)))

class_colors = {
    1: (255, 50, 50),    # fiber  = red
    2: (50, 255, 50),    # film   = green
    3: (50, 50, 255),    # fragment = blue
}

fig, axes = plt.subplots(len(test_samples), 4,
                         figsize=(20, 4 * len(test_samples)))
if len(test_samples) == 1:
    axes = axes.reshape(1, -1)

fig.suptitle('Predictions vs SAM Ground Truth', fontsize=14, y=1.01)
for col, title in enumerate(['Original', 'SAM GT Mask',
                              'Predicted Mask', 'Prediction Overlay']):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

for i, name in enumerate(test_samples):
    ann = test_anns[name]

    img = cv2.imread(str(sam_path / 'images' / name))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Col 0: Original
    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_ylabel(ann['class_name'], fontsize=10,
                          rotation=0, labelpad=50)
    axes[i, 0].axis('off')

    # Col 1: SAM ground truth
    mask_file = ann.get('mask_file', name.replace('.png', '_mask.png'))
    gt_path = sam_path / 'masks' / mask_file
    if gt_path.exists():
        gt_mask = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
        gt_binary = (gt_mask > 127).astype(np.uint8)
        gt_overlay = img_rgb.copy()
        gt_overlay[gt_binary == 1] = (
            gt_overlay[gt_binary == 1] * 0.5 +
            np.array([0, 255, 0]) * 0.5
        ).astype(np.uint8)
        axes[i, 1].imshow(gt_overlay)
    axes[i, 1].axis('off')

    # Inference
    img_tensor = inference_transform(img_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        predictions = model(img_tensor)[0]

    score_thresh = 0.3
    keep = predictions['scores'] > score_thresh

    if keep.sum() > 0:
        pred_masks = predictions['masks'][keep]
        pred_labels = predictions['labels'][keep]
        pred_scores = predictions['scores'][keep]
        pred_boxes = predictions['boxes'][keep]

        best_mask = pred_masks[0, 0].cpu().numpy()
        best_label = pred_labels[0].item()
        best_score = pred_scores[0].item()
        best_box = pred_boxes[0].cpu().numpy().astype(int)
        pred_class = (CLASS_NAMES[best_label]
                      if best_label < len(CLASS_NAMES) else '?')

        # Col 2: Predicted mask
        axes[i, 2].imshow(best_mask > 0.5, cmap='gray')
        axes[i, 2].set_title(f"{pred_class} ({best_score:.2f})", fontsize=9)
        axes[i, 2].axis('off')

        # Col 3: Overlay + bbox
        img_resized = cv2.resize(img_rgb, (CROP_SIZE, CROP_SIZE))
        pred_binary = (best_mask > 0.5).astype(np.uint8)
        color = class_colors.get(best_label, (255, 255, 0))
        color_norm = np.array(color) / 255.0

        overlay = img_resized.copy().astype(np.float32) / 255.0
        overlay[pred_binary == 1] = (
            overlay[pred_binary == 1] * 0.5 + color_norm * 0.5)

        x1, y1, x2, y2 = best_box
        overlay_uint8 = (overlay * 255).astype(np.uint8)
        cv2.rectangle(overlay_uint8, (x1, y1), (x2, y2), color, 2)
        cv2.putText(overlay_uint8, f"{pred_class} {best_score:.2f}",
                    (x1, max(y1-5, 10)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.35, color, 1)

        axes[i, 3].imshow(overlay_uint8)
        axes[i, 3].axis('off')
    else:
        axes[i, 2].text(0.5, 0.5, 'No detection',
                       ha='center', va='center', fontsize=12)
        axes[i, 2].axis('off')
        axes[i, 3].text(0.5, 0.5, 'No detection',
                       ha='center', va='center', fontsize=12)
        axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

## 11. Save Models to Google Drive

Copy **only** the trained model checkpoints back to Drive.

In [ ]:
DRIVE_EXPERIMENTS.mkdir(parents=True, exist_ok=True)

for model_file in ['maskrcnn_crops_best.pth', 'maskrcnn_crops_latest.pth']:
    src = SAVE_DIR / model_file
    dst = DRIVE_EXPERIMENTS / model_file
    if src.exists():
        shutil.copy2(str(src), str(dst))
        size_mb = src.stat().st_size / 1024 / 1024
        print(f"\u2713 {model_file} ({size_mb:.1f} MB) \u2192 Drive")
    else:
        print(f"\u2717 {model_file} not found locally")

print(f"\n{'='*60}")
print("MODELS SAVED TO GOOGLE DRIVE")
print(f"{'='*60}")
print(f"  Best:   {DRIVE_EXPERIMENTS / 'maskrcnn_crops_best.pth'}")
print(f"  Latest: {DRIVE_EXPERIMENTS / 'maskrcnn_crops_latest.pth'}")
print(f"{'='*60}")

## Done!

**Models saved to:** `MyDrive/mp-detect/experiments/`
- `maskrcnn_crops_best.pth` — best validation loss
- `maskrcnn_crops_latest.pth` — last epoch

**Anti-overfit measures applied (51 train + 11 val → ~10K augmented crops):**
- **Train/Val split by source image** — val crops never seen during training
- **Phase 1**: backbone frozen (5 epochs, LR=5e-4) — warm up heads safely
- **Phase 2**: full fine-tune (20 epochs, LR=1e-4) — careful end-to-end
- **ReduceLROnPlateau** — LR halved when val loss stalls (patience 3)
- **Early stopping** (patience 7) — prevents training past best generalisation
- **Strong augmentation**: elastic, affine, colour jitter, cutout, blur
- **High weight decay** (0.005) — L2 regularisation on all parameters

**Use in the full pipeline:**
```bash
python src/pipeline_inference.py \
    --input dev-test/stitched/s1.png \
    --yolo experiments/augmented_microplastic_yolo/weights/best.pt \
    --effnet experiments/efficientnet_best.pth \
    --maskrcnn experiments/maskrcnn_crops_best.pth
```